In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
SILVER_TABLE = "traffic_project.silver.track_events"
GOLD_TABLE = "traffic_project.gold.turning_movements"

In [0]:

silver_df = spark.table(SILVER_TABLE)

In [0]:
gold_df = (
    silver_df
    .groupBy("camera_id", "track_id")
    .agg(
        F.min_by("roi_zone", "event_time").alias("entry_zone"),
        F.max_by("roi_zone", "event_time").alias("exit_zone"),
        F.min("event_time").alias("entry_time"),
        F.max("event_time").alias("exit_time"),
        F.max(
            F.when(F.col("track_status") == "lost", 1).otherwise(0)
        ).alias("had_lost_track")
    )
    .filter(
        (F.col("had_lost_track") == 0)
        & F.col("entry_zone").rlike("_entry$")
        & F.col("exit_zone").rlike("_exit$")
        & (F.col("exit_time") > F.col("entry_time"))
    )
    .withColumn(
        "movement",
        F.concat_ws(" -> ", "entry_zone", "exit_zone")
    )
    .withColumn("movement_status", F.lit("confirmed"))
)

In [0]:
gold_df.createOrReplaceTempView("gold_updates")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS traffic_project.gold.turning_movements
USING DELTA
AS
SELECT *
FROM gold_updates
WHERE 1 = 0
""")

In [0]:
spark.sql("""
MERGE INTO traffic_project.gold.turning_movements AS target
USING gold_updates AS source
ON  target.camera_id = source.camera_id
AND target.track_id = source.track_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
""")

In [0]:
%sql
SELECT * FROM traffic_project.gold.turning_movements